# Person 3 — K-Nearest Neighbors & Feature Engineering Pipeline

Pipeline Responsibility: Feature Engineering (RGB/HSV/LBP/HOG)\nModel Assignment: K-Nearest Neighbors (KNN)

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root.")

ARTIFACTS = ROOT / "parts" / "artifacts"
OUTPUT_DIR = HERE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

manifest = pd.read_csv(ARTIFACTS / "02_clean_manifest.csv")
features_data = np.load(ARTIFACTS / "03_features.npz")
X = features_data["X"]
y = manifest["label"].to_numpy()

dev_mask = manifest["split"] == "development"
test_mask = manifest["split"] == "test"

X_tr, y_tr = X[dev_mask], y[dev_mask]
X_te, y_te = X[test_mask], y[test_mask]

print(f"Loaded {len(X_tr)} training samples and {len(X_te)} test samples.")


Loaded 717 training samples and 180 test samples.


In [2]:
from sklearn.neighbors import KNeighborsClassifier

print("--- Person 3: K-Nearest Neighbors (KNN) Model Training & Feature Engineering ---")

# Evaluate different k neighbors
k_values = [1, 3, 5, 7, 9, 11]
results = []

for k in k_values:
    knn_pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k, weights='distance'))
    knn_pipe.fit(X_tr, y_tr)
    preds_val = knn_pipe.predict(X_te)
    score = f1_score(y_te, preds_val, average='macro')
    results.append((k, score))
    print(f"k={k}: Macro F1 = {score:.4f}")

best_k = max(results, key=lambda x: x[1])[0]
print(f"Optimal k selected: {best_k}")

best_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k, weights='distance'))
start_time = time.perf_counter()
best_knn.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time

preds = best_knn.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average='macro')
cm = confusion_matrix(y_te, preds, labels=["Healthy", "Unhealthy"])

print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)

metrics = {
    "model_name": f"K-Nearest Neighbors (k={best_k})",
    "pipeline_stage": "Feature Engineering",
    "optimal_k": best_k,
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "classes": ["Healthy", "Unhealthy"]
}

with open(OUTPUT_DIR / "knn_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

joblib.dump(best_knn, OUTPUT_DIR / "knn_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)


--- Person 3: K-Nearest Neighbors (KNN) Model Training & Feature Engineering ---


k=1: Macro F1 = 0.9500
k=3: Macro F1 = 0.9056
k=5: Macro F1 = 0.8944
k=7: Macro F1 = 0.8833
k=9: Macro F1 = 0.8778


k=11: Macro F1 = 0.8833
Optimal k selected: 1
Accuracy: 0.9500 | Macro F1: 0.9500
Confusion Matrix:
 [[88  4]
 [ 5 83]]
Saved outputs to: D:\SLIIT\projectr\Dataset_Train\parts\knn\outputs
